# 01. 탐색적 데이터 분석 (EDA)
## 개인 소비 분석 + 무의식 지출 탐지 AI

**목표:**
- 소비 데이터 기본 구조 파악
- 카테고리별 / 요일별 / 시간대별 지출 패턴 시각화
- 충동 소비 가능성 높은 패턴 1차 파악

## 0. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료!')

## 1. 데이터 로드 및 기본 확인

In [ ]:
df = pd.read_csv('../data/raw/sample_spending.csv')

# 날짜/시간 변환
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
df['date'] = pd.to_datetime(df['date'])
df['hour'] = pd.to_datetime(df['time'], format='%H:%M').dt.hour

# 요일 이름 추가
weekday_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
df['weekday_name'] = df['weekday'].map(weekday_map)

print(f'데이터 shape: {df.shape}')
df.head(10)

In [ ]:
print('=== 기본 정보 ===')
print(df.info())
print('\n=== 결측값 ===')
print(df.isnull().sum())
print('\n=== 기초 통계 ===')
df['amount'].describe()

## 2. 전체 지출 현황

In [ ]:
total = df['amount'].sum()
avg_daily = df.groupby('date')['amount'].sum().mean()
avg_per_tx = df['amount'].mean()

print(f'총 지출:       {total:,.0f}원')
print(f'일 평균 지출:  {avg_daily:,.0f}원')
print(f'건당 평균:     {avg_per_tx:,.0f}원')
print(f'총 거래 건수:  {len(df)}건')

## 3. 카테고리별 지출 분석

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 카테고리별 합계
cat_sum = df.groupby('category')['amount'].sum().sort_values(ascending=False)

# 막대 그래프
axes[0].bar(cat_sum.index, cat_sum.values, color='steelblue', edgecolor='white')
axes[0].set_title('카테고리별 총 지출', fontsize=14)
axes[0].set_ylabel('금액 (원)')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(cat_sum.values):
    axes[0].text(i, v + 500, f'{v:,.0f}', ha='center', fontsize=9)

# 파이 차트
axes[1].pie(cat_sum.values, labels=cat_sum.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('카테고리별 지출 비율', fontsize=14)

plt.tight_layout()
plt.savefig('../data/processed/01_category_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. 요일별 지출 패턴

In [ ]:
weekday_order = ['월', '화', '수', '목', '금', '토', '일']
weekday_sum = df.groupby('weekday_name')['amount'].sum().reindex(weekday_order)
weekday_cnt = df.groupby('weekday_name')['amount'].count().reindex(weekday_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#FF6B6B' if w in ['토', '일'] else '#4ECDC4' for w in weekday_order]

axes[0].bar(weekday_order, weekday_sum.values, color=colors, edgecolor='white')
axes[0].set_title('요일별 총 지출금액', fontsize=14)
axes[0].set_ylabel('금액 (원)')
for i, v in enumerate(weekday_sum.values):
    axes[0].text(i, v + 500, f'{v:,.0f}', ha='center', fontsize=8)

axes[1].bar(weekday_order, weekday_cnt.values, color=colors, edgecolor='white')
axes[1].set_title('요일별 거래 건수', fontsize=14)
axes[1].set_ylabel('건수')

plt.tight_layout()
plt.savefig('../data/processed/02_weekday_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. 시간대별 지출 패턴 (충동 소비 힌트)

In [ ]:
hour_sum = df.groupby('hour')['amount'].sum()
hour_cnt = df.groupby('hour')['amount'].count()

# 심야 시간대 (22시~03시) 강조
colors = ['#FF4444' if h >= 22 or h <= 3 else '#5B8CFF' for h in hour_sum.index]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(hour_sum.index, hour_sum.values, color=colors, edgecolor='white')
ax.set_title('시간대별 지출금액 (빨간색 = 심야 충동 소비 위험)', fontsize=13)
ax.set_xlabel('시간 (24h)')
ax.set_ylabel('금액 (원)')
ax.set_xticks(range(24))

from matplotlib.patches import Patch
legend = [Patch(color='#FF4444', label='심야 (22시 이후)'),
          Patch(color='#5B8CFF', label='일반 시간대')]
ax.legend(handles=legend)

plt.tight_layout()
plt.savefig('../data/processed/03_hour_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

# 심야 지출 요약
night_spend = df[df['hour'] >= 22]['amount'].sum()
total_spend = df['amount'].sum()
print(f'심야(22시~) 총 지출: {night_spend:,.0f}원 ({night_spend/total_spend*100:.1f}%)')

## 6. 일별 지출 추이 (시계열)

In [ ]:
daily = df.groupby('date')['amount'].sum().reset_index()
daily['rolling_7'] = daily['amount'].rolling(7).mean()

fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(daily['date'], daily['amount'], alpha=0.5, color='steelblue', label='일별 지출')
ax.plot(daily['date'], daily['rolling_7'], color='red', linewidth=2, label='7일 이동평균')
ax.set_title('일별 지출 추이', fontsize=14)
ax.set_xlabel('날짜')
ax.set_ylabel('금액 (원)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/processed/04_daily_trend.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. 충동 소비 의심 거래 추출
> 기준: 심야(22시 이후) + 온라인 쇼핑 + 메모에 감정 단어 포함

In [ ]:
# 충동 소비 키워드
impulse_keywords = ['스트레스', '충동', '답답', '피곤', '그냥', '질러', '기분', '위로', '폭식', '새벽']

def has_impulse_keyword(memo):
    if pd.isna(memo) or memo == '':
        return False
    return any(kw in str(memo) for kw in impulse_keywords)

# 충동 소비 의심 플래그
df['is_impulse_memo'] = df['memo'].apply(has_impulse_keyword)
df['is_night'] = df['hour'] >= 22
df['is_impulse'] = df['is_impulse_memo'] | (df['is_night'] & (df['category'] == '쇼핑'))

impulse_df = df[df['is_impulse'] == True][['date', 'time', 'category', 'amount', 'memo']]

print(f'충동 소비 의심 건수: {len(impulse_df)}건')
print(f'충동 소비 의심 금액: {impulse_df["amount"].sum():,.0f}원')
print(f'전체 대비 비율: {impulse_df["amount"].sum() / df["amount"].sum() * 100:.1f}%\n')
impulse_df

## 8. EDA 요약

In [ ]:
print('=' * 50)
print('EDA 요약 리포트')
print('=' * 50)
print(f'분석 기간:         {df["date"].min().date()} ~ {df["date"].max().date()}')
print(f'총 지출:           {df["amount"].sum():,.0f}원')
print(f'총 거래 건수:      {len(df)}건')
print(f'일 평균 지출:      {df.groupby("date")["amount"].sum().mean():,.0f}원')
print(f'최대 단일 지출:    {df["amount"].max():,.0f}원 ({df.loc[df["amount"].idxmax(), "category"]})')
print(f'가장 많이 쓴 카테고리: {df.groupby("category")["amount"].sum().idxmax()}')
print(f'심야 지출 비율:    {df[df["hour"] >= 22]["amount"].sum() / df["amount"].sum() * 100:.1f}%')
print(f'충동 소비 의심:    {impulse_df["amount"].sum():,.0f}원 ({impulse_df["amount"].sum() / df["amount"].sum() * 100:.1f}%)')
print('=' * 50)
print('\n다음 단계: 02_emotion_analysis.ipynb (NLP 감정 분석)')

In [ ]:
# 전처리된 데이터 저장
df.to_csv('../data/processed/spending_processed.csv', index=False, encoding='utf-8-sig')
print('전처리 데이터 저장 완료: data/processed/spending_processed.csv')